In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.coordinates import Galactic, ICRS
from astropy import units as u
from modules.vr_opt import VrOpt

import time
import random
import nbimporter
import visualisations
import consensus_clusterings

from sklearn.preprocessing import LabelEncoder
from matplotlib.patches import Polygon

import sys

from sklearn.neighbors import NearestNeighbors
from sklearn.mixture import GaussianMixture
import itertools

from collections import defaultdict
from tqdm import tqdm 


In [16]:
sys.path.append('/Users/lui/Documents/3_semestr/Project/code/local-clustering/SemiMetricClustering')
from SemiMetricDensity import SemiMetricDensity

In [7]:
def rotate_dataset(df, angle_deg, axis='z'):
    angle_rad = np.radians(angle_deg)
    if axis == 'x':
        rotation_matrix = np.array([
            [1, 0, 0],
            [0, np.cos(angle_rad), -np.sin(angle_rad)],
            [0, np.sin(angle_rad), np.cos(angle_rad)]
        ])
    elif axis == 'y':
        rotation_matrix = np.array([
            [np.cos(angle_rad), 0, np.sin(angle_rad)],
            [0, 1, 0],
            [-np.sin(angle_rad), 0, np.cos(angle_rad)]
        ])
    elif axis == 'z':
        rotation_matrix = np.array([
            [np.cos(angle_rad), -np.sin(angle_rad), 0],
            [np.sin(angle_rad), np.cos(angle_rad), 0],
            [0, 0, 1]
        ])
    else:
        raise ValueError("Axis must be 'x', 'y', or 'z'.")

    points = df[['x', 'y', 'z']].values
    rotated_points = points @ rotation_matrix.T
    rotated_df = df.copy()
    rotated_df[['x', 'y', 'z']] = rotated_points
    return rotated_df

In [8]:
# Shift grid by vector
def shift_grid(x_range, y_range, z_range, shift_vector):
   
    dx, dy, dz = shift_vector
    return x_range + dx, y_range + dy, z_range + dz

In [9]:
# Create extended grid to ensure it covers all points, even after shifts
def create_grid(df, grid_size, shift_vector, num_shifts):

    # min and max ranges for x, y, z
    x_min, x_max = df['x'].min(), df['x'].max()
    y_min, y_max = df['y'].min(), df['y'].max()
    z_min, z_max = df['z'].min(), df['z'].max()

    # Maximum extent required for grid
    dx, dy, dz = shift_vector
    x_min_extended = x_min - abs(dx) * num_shifts
    x_max_extended = x_max + abs(dx) * num_shifts
    y_min_extended = y_min - abs(dy) * num_shifts
    y_max_extended = y_max + abs(dy) * num_shifts
    z_min_extended = z_min - abs(dz) * num_shifts
    z_max_extended = z_max + abs(dz) * num_shifts

    # Grid ranges
    x_range = np.arange(x_min_extended, x_max_extended + grid_size, grid_size)
    y_range = np.arange(y_min_extended, y_max_extended + grid_size, grid_size)
    z_range = np.arange(z_min_extended, z_max_extended + grid_size, grid_size)

    return x_range, y_range, z_range

In [10]:
def plot_grid_with_dataset(df, x_range, y_range, z_range, true_labels, colors, alphas, zorders, projection='xy', title="Dataset with grid"):

    # Select axes based on projection
    if projection == 'xy':
        points = df[['x', 'y']].values
        grid_x = x_range
        grid_y = y_range
        axis_labels = ('X-axis', 'Y-axis')
    elif projection == 'xz':
        points = df[['x', 'z']].values
        grid_x = x_range
        grid_y = z_range
        axis_labels = ('X-axis', 'Z-axis')
    elif projection == 'yz':
        points = df[['y', 'z']].values
        grid_x = y_range
        grid_y = z_range
        axis_labels = ('Y-axis', 'Z-axis')
    else:
        raise ValueError("Projection must be one of 'xy', 'xz', 'yz'")

    # labels to integers
    true_labels = true_labels.astype(int)

    # Background
    background_color = colors[-1]
    background_alpha = alphas[-1]
    background_zorder = zorders[-1]

    # Clusters
    cluster_colors = colors[:-1]
    cluster_alphas = alphas[:-1]
    cluster_zorders = zorders[:-1]

    # Map labels
    unique_labels = np.unique(true_labels)
    label_to_color = {}
    label_to_alpha = {}
    label_to_zorder = {}

    for label in unique_labels:
        if label == 0:  # Background
            label_to_color[label] = background_color
            label_to_alpha[label] = background_alpha
            label_to_zorder[label] = background_zorder
        else:  # Clusters
            cluster_idx = (label - 1) % len(cluster_colors)
            label_to_color[label] = cluster_colors[cluster_idx]
            label_to_alpha[label] = cluster_alphas[cluster_idx]
            label_to_zorder[label] = cluster_zorders[cluster_idx]

    fig, ax = plt.subplots(figsize=(10, 10))

    # Dataset points
    for label in unique_labels:
        idx = true_labels == label
        ax.scatter(
            points[idx, 0],
            points[idx, 1],
            s=10,
            color=label_to_color[label],
            alpha=label_to_alpha[label],
            zorder=label_to_zorder[label],
            label=f'Label {label}'
        )

    # Grid lines
    for x in grid_x:
        ax.plot([x, x], [grid_y[0], grid_y[-1]], color='black', alpha=0.5, linestyle='--', linewidth=1, zorder=0)
    for y in grid_y:
        ax.plot([grid_x[0], grid_x[-1]], [y, y], color='black', alpha=0.5, linestyle='--', linewidth=1, zorder=0)

    ax.set_title(title, fontsize=16)
    ax.set_xlabel(axis_labels[0])
    ax.set_ylabel(axis_labels[1])
    ax.legend(fontsize=12)
    plt.show()

In [36]:
def partition_points_into_grid(df, x_range, y_range, z_range):
    grid_partitions = defaultdict(list)

    for idx, row in df.iterrows():
        x, y, z = row['x'], row['y'], row['z']
        
        # Find the correct grid cell
        x_idx = np.searchsorted(x_range, x, side="right") - 1
        y_idx = np.searchsorted(y_range, y, side="right") - 1
        z_idx = np.searchsorted(z_range, z, side="right") - 1
        
        # Ensure indices are within range
        x_idx = max(0, min(x_idx, len(x_range) - 2))
        y_idx = max(0, min(y_idx, len(y_range) - 2))
        z_idx = max(0, min(z_idx, len(z_range) - 2))

        cell = (x_idx, y_idx, z_idx)
        grid_partitions[cell].append(idx)

    return grid_partitions


In [37]:
def run_square_grid_consensus_clustering(df, grid_size, shift_vector, K, true_labels, solver,
                                         consensus_nclass, random_state, colors, alphas, zorders,
                                         num_rotations=3, rotation_axis='z', rotation_angle=None, num_shifts=3,
                                         projection='xy', max_iter=1_000, method="random", theta=0.001, epsilon=0.0001,
                                         initial_partition=None, rho_multiplier=None, verbose=False, th_dist=None, dist_max=1e3,
                                         print_info = False, entire_G=False):
 
    grid_results = []
    original_dataset_size = len(df)

    smc = SemiMetricDensity(data=df, K=K, max_iter=max_iter, method=method)

    # Calculate entire G matrix at beginning
    if entire_G:
        smc.compute_G(th_dist=th_dist, dist_max=dist_max) 
        G = smc.G 

    if rotation_angle is None:
        rotation_angle = 360 / num_rotations if num_rotations > 1 else 0

    for rotation_step in range(num_rotations):
        if print_info == True:
            print(f"Rotation {rotation_step + 1}: {'No rotation' if rotation_step == 0 else f'Rotating by {rotation_step * rotation_angle:.2f}° around {rotation_axis}-axis'}")
        rotated_df = df.copy() if rotation_step == 0 else rotate_dataset(df, rotation_step * rotation_angle, axis=rotation_axis)
        rotated_indices = np.arange(original_dataset_size)

        # Initialize grid ranges
        x_range, y_range, z_range = create_grid(rotated_df, grid_size, shift_vector, num_shifts)

        for shift_step in range(num_shifts):
            if print_info == True:
                print(f"  Shift {shift_step + 1}: Applying shift vector {shift_vector}")

            label_offset = 0  # Reset label offset for each shift
            grid_partitions = partition_points_into_grid(rotated_df, x_range, y_range, z_range)
            shift_results = []

            for cell_idx, (cell, indices) in enumerate(grid_partitions.items()):
                if len(indices) == 0:
                    continue  # Skip empty cells

                original_indices = rotated_indices[indices]

                # Ensure valid indices
                original_indices = original_indices[original_indices < original_dataset_size]
                if len(original_indices) == 0:
                    continue  # Skip empty indices

                # Ensure compute_G() is only called when valid indices exist
                if entire_G:
                    smc.G = G[np.ix_(original_indices, original_indices)]
                else:
                    if len(original_indices) > 1:  # Ensure at least two points for distances
                        smc.compute_G(original_indices, th_dist, dist_max)
                    else:
                        continue  # Skip computation if not enough points

                # Perform clustering
                p_i, _ = smc.run_softmax(theta=theta, epsilon=epsilon, initial_partition=initial_partition, rho_multiplier=rho_multiplier, verbose=verbose)

                # Assign clusters
                softmax_cluster_labels = np.argmax(p_i, axis=1)
                unique_seq_labels = np.unique(softmax_cluster_labels)

                label_mapping = {old_label: new_label + label_offset for new_label, old_label in enumerate(unique_seq_labels)}
                cluster_labels = np.array([label_mapping[label] for label in softmax_cluster_labels])

                label_offset += len(unique_seq_labels)

                shift_results.append((cluster_labels, original_indices))

                if print_info:
                    print(f"    Rotation {rotation_step + 1}, Shift {shift_step + 1}, Cell {cell_idx + 1}:")
                    print(f"      Number of clusters: {len(unique_seq_labels)}")
                    print(f"      Cluster labels: {np.unique(cluster_labels)}")

            grid_results.append((rotation_step, shift_step, shift_results))

            # Visualization check
            if print_info == True: 
                plot_grid_with_dataset(
                    rotated_df, x_range, y_range, z_range, true_labels, colors, alphas, zorders,
                    projection=projection, title=f"Rotation {rotation_step + 1}, Grid Shift {shift_step + 1}"
                )

            # Shift grid for next iteration
            x_range, y_range, z_range = shift_grid(x_range, y_range, z_range, shift_vector)

    consensus_results = consensus_clusterings.consensus_clustering_from_grid(grid_results, solver=solver, nclass=consensus_nclass, random_state=random_state, verbose=verbose)

    return consensus_results

In [41]:
def rerun_square_grid_clustering_on_detected_cluster(df, true_labels, detected_cluster_id, consensus_labels, 
                                                    K, solver, consensus_nclass, num_rotations, shift_vector, num_shifts, 
                                                    grid_size, projection, colors, alphas, zorders, 
                                                    rotation_axis='z', rotation_angle=None, max_iter=1000, method="random", 
                                                    theta=0.001, epsilon=0.0001, th_dist=None, dist_max=1000, 
                                                    print_info=False, entire_G=False):
    
    # Select points belonging only to the specified detected cluster
    cluster_indices = np.where(consensus_labels == detected_cluster_id)[0]
    subset_df = df.iloc[cluster_indices].reset_index(drop=True)
    
    if isinstance(true_labels, pd.Series):
        subset_true_labels = true_labels.iloc[cluster_indices].reset_index(drop=True)
    else:
        subset_true_labels = true_labels[cluster_indices]

    if len(subset_df) < 2:
        print(f"Cluster {detected_cluster_id}: Not enough points to rerun clustering.")
        return None, None, None, None
    
    if print_info == True:
        print(f"Rerunning clustering on detected cluster {detected_cluster_id} with {len(subset_df)} points.")

    # Run shell grid consensus clustering on the subset
    start_time = time.time()
    clustering_results = run_square_grid_consensus_clustering(
        subset_df, grid_size, shift_vector, K, subset_true_labels, solver, consensus_nclass,
        random_state=42, colors=colors, alphas=alphas, zorders=zorders,
        num_rotations=num_rotations, rotation_axis=rotation_axis, rotation_angle=rotation_angle,
        num_shifts=num_shifts, projection=projection, max_iter=max_iter, method=method,
        theta=theta, epsilon=epsilon, initial_partition=None,
        rho_multiplier=None, verbose=False, th_dist=th_dist, 
        dist_max=dist_max, print_info=print_info, entire_G=entire_G
    )
    clustering_time = time.time() - start_time

    # Ensure the true labels match the length of new_consensus_labels
    subset_true_labels = subset_true_labels[:len(clustering_results)]

    return clustering_results, subset_df, subset_true_labels, clustering_time

In [54]:
def test_square_grid_clustering(df, true_labels, grid_sizes, shift_vectors, K_values, solvers, 
                                consensus_nclass_values, num_rotations_values, rotation_angles, 
                                num_shifts_values, max_iter_values, theta_values, epsilon_values, 
                                th_dist_values, dist_max_values, colors, alphas, zorders, output_csv="clustering_results.csv"):
    
    # Generate all parameter combinations
    param_combinations = list(itertools.product(
        grid_sizes, shift_vectors, K_values, solvers, consensus_nclass_values,
        num_rotations_values, rotation_angles, num_shifts_values, max_iter_values,
        theta_values, epsilon_values, th_dist_values, dist_max_values
    ))
    
    # Store results
    results = []
    
    # Initialize progress bar
    with tqdm(total=len(param_combinations), desc="Testing Square Grid Clustering") as pbar:
        for i, params in enumerate(param_combinations):
            (grid_size, shift_vector, K, solver, consensus_nclass,
             num_rotations, rotation_angle, num_shifts, max_iter,
             theta, epsilon, th_dist, dist_max) = params
            
            start_time = time.time()

            consensus_labels = run_square_grid_consensus_clustering(
                    df=df,
                    grid_size=grid_size,
                    shift_vector=shift_vector,
                    K=K,
                    true_labels=true_labels,
                    solver=solver,
                    consensus_nclass=consensus_nclass,
                    random_state=42,
                    colors=colors,
                    alphas=alphas,
                    zorders=zorders,
                    num_rotations=num_rotations,
                    rotation_angle=rotation_angle,
                    num_shifts=num_shifts,
                    max_iter=max_iter,
                    method="random",
                    theta=theta,
                    epsilon=epsilon,
                    th_dist=th_dist,
                    dist_max=dist_max,
                    print_info=False,
                    entire_G=False
                )

            clustering_time = time.time() - start_time
            num_clusters = len(set(consensus_labels))
            nmi = consensus_clusterings.calculate_nmi(consensus_labels, true_labels)

            results.append({
                    "Grid Size": grid_size, "Shift Vector": shift_vector, "K": K, "Solver": solver,
                    "Consensus Nclass": consensus_nclass, "Num Rotations": num_rotations, "Rotation Angle": rotation_angle,
                    "Num Shifts": num_shifts, "Max Iter": max_iter, "Theta": theta, "Epsilon": epsilon,
                    "Th_dist": th_dist, "Dist_max": dist_max,  "Num Clusters": num_clusters, "NMI": nmi, "Runtime (s)": clustering_time
                })
            
            # Update progress bar
            pbar.update(1)
    
    results_df = pd.DataFrame(results)

    # Save results to CSV
    results_df.to_csv(output_csv, index=False)
       
    return results